In [ ]:
from __future__ import annotations


def main(datasources, start_date, end_date):
    """Return the reversal of standardized intraday closing displacement."""
    import numpy as np
    import pandas as pd
    import dai

    output_columns = ["date", "instrument", "factor"]
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    start_day = start.normalize()
    end_day = end.normalize()

    # The factor itself has no rolling window.  The historical buffer is used
    # only as a causal fallback when a self-test table omits the last day's
    # minute bars but the official universe already contains that date.
    query_start = start_day - pd.Timedelta(days=30)
    bar1m = datasources["bar1m"]
    bars = dai.query(
        f"""
        SELECT
            CAST(date_trunc('day', date) AS TIMESTAMP) AS date,
            instrument,
            arg_max(close, date) AS close,
            avg(CAST(close AS DOUBLE)) AS minute_mean,
            stddev_samp(CAST(close AS DOUBLE)) AS minute_std
        FROM {bar1m}
        WHERE close > 0
        GROUP BY 1, 2
        ORDER BY 2, 1
        """,
        filters={"date": [query_start, end]},
        compression=True,
    ).df()

    if bars.empty:
        signals = pd.DataFrame(columns=output_columns)
    else:
        bars["date"] = pd.to_datetime(bars["date"]).dt.normalize()
        bars["instrument"] = bars["instrument"].astype(str)
        value_columns = ["close", "minute_mean", "minute_std"]
        for column in value_columns:
            bars[column] = pd.to_numeric(bars[column], errors="coerce")
        bars = (
            bars.sort_values(["instrument", "date"])
            .drop_duplicates(["date", "instrument"], keep="last")
            .reset_index(drop=True)
        )

        valid_price = (
            (bars["close"] > 0.0)
            & (bars["minute_mean"] > 0.0)
            & (bars["minute_std"] > 0.0)
        )
        bars.loc[~valid_price, value_columns] = np.nan

        # A positive closing_z means the close is unusually high relative to
        # that stock's own intraday path.  Cross-sectional percentile ranking
        # makes different price/volatility scales comparable.  The minus sign
        # expresses the next-day reversal hypothesis.
        closing_z = (bars["close"] - bars["minute_mean"]) / bars["minute_std"]
        displacement_rank = closing_z.groupby(
            bars["date"], sort=False
        ).rank(pct=True, method="average")
        raw_factor = -displacement_rank
        bars["factor"] = raw_factor - raw_factor.groupby(
            bars["date"], sort=False
        ).transform("mean")
        bars["factor"] = bars["factor"].replace([np.inf, -np.inf], np.nan)
        signals = bars.loc[:, output_columns].dropna(subset=["factor"])

    # The fixed official pool is the documented fallback.  Supplying an
    # instruments mapping remains useful for local/offline verification.
    instruments_table = datasources.get(
        "instruments", "bigalpha_2026_instruments"
    )
    universe = dai.query(
        f"SELECT date, instrument FROM {instruments_table}",
        filters={"date": [start, end]},
        compression=True,
    ).df()
    universe["date"] = pd.to_datetime(universe["date"]).dt.normalize()
    universe["instrument"] = universe["instrument"].astype(str)
    universe = (
        universe.loc[universe["date"].between(start_day, end_day)]
        .drop_duplicates(["date", "instrument"], keep="last")
        .sort_values(["instrument", "date"])
        .reset_index(drop=True)
    )

    target = universe.merge(
        signals, on=["date", "instrument"], how="left", validate="one_to_one"
    )
    target["_target"] = True

    # Causal tail-day fallback: bring in only the latest signal strictly before
    # the requested interval, then forward-fill per stock.  The fallback is
    # enabled only when the minute table lacks an entire universe date; ordinary
    # stock-level missing values retain the original factor's zero-fill behavior.
    # No future value is ever back-filled.
    seed = (
        signals.loc[signals["date"] < start_day]
        .sort_values(["instrument", "date"])
        .drop_duplicates("instrument", keep="last")
        .copy()
    )
    seed["_target"] = False
    combined = pd.concat([seed, target], ignore_index=True, sort=False)
    combined = combined.sort_values(["instrument", "date", "_target"])
    combined["_causal_factor"] = combined.groupby(
        "instrument", sort=False
    )["factor"].ffill()
    covered_dates = set(signals["date"].unique())
    covered_target = combined["_target"] & combined["date"].isin(covered_dates)
    combined.loc[covered_target, "_causal_factor"] = combined.loc[
        covered_target, "factor"
    ]
    result = combined.loc[
        combined["_target"], ["date", "instrument", "_causal_factor"]
    ].rename(columns={"_causal_factor": "factor"})
    result["factor"] = pd.to_numeric(result["factor"], errors="coerce")
    result["factor"] = (
        result["factor"].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    )
    return result.sort_values(["date", "instrument"]).reset_index(drop=True)
